## 01\_eda.ipynb: Steps 1 &amp; 2 — Business Understanding, Data Ingestion &amp; Preprocessing

**Course**: ADY201m — AI &amp; Data Science with Python &amp; SQL (Fall 2026) **Project Title**: Multimodal Classification and Fast Small-model Summarisation of UAS Incident Reports from NASA ASRS for Safe Aviation Infrastructure **Group**: Group 6 / Group 7 (Class AI2108) | **Role**: Student A (Steps 0–3) **UN Sustainable Development Goal**: **SDG 9 — Resilient Infrastructure &amp; Innovation**

---

### 1\. Business Understanding &amp; Project Context

* **Problem Statement**: Unmanned Aircraft System (UAS) incidents represent a rapidly growing threat to safe airspace and aviation infrastructure. NASA Aviation Safety Reporting System (ASRS) reports contain valuable safety intelligence split across structured metadata (flight phase, weather, airspace) and unstructured text narratives (`Narrative` / `Synopsis`).
* **Analytic Approach**: Multimodal Multi-class Classification &amp; Small Language Model (SLM) Fast Summarization.
* **Target Variable (** **y** **)**: Incident Category mapped from raw ASRS `Anomaly` into 5 classes:
  1. `deviation` (Flight/Altitude/Speed Deviations)
  2. `aircraft equipment` (Equipment/System Failures)
  3. `conflict` (Airborne/Ground Proximity Risks)
  4. `airspace violation` (Unauthorized Airspace Penetration)
  5. `other` (General operational anomalies)

---

### 2\. Core Research Questions (RQs)

* **RQ1 (Multimodal Classification)**: Does combining structured metadata and unstructured text narratives improve UAS incident classification accuracy and F1-score compared to single-modality models?
* **RQ2 (Fast SLM Summarization)**: Can Small Language Models (SLMs) generate concise incident summaries while preserving critical safety information (ROUGE-1/2/L, Unsupported Entity Rate)?
* **RQ3 (Quality vs. Speed Trade-off)**: What is the trade-off between summary quality and inference speed/memory footprint when deploying lightweight SLMs using speculative decoding?

---

### 3\. Week 1–3 Scope &amp; Output Deliverables

* **Data Ingestion**: Load `ASRS_FULL_DATASET.csv` (&gt;125k records, 127 columns) via DuckDB.
* **Filtering &amp; Preprocessing**: Filter for UAS records (`is_uas`), drop short/empty narratives (&lt;100 chars), handle missing values, and deduplicate IDs.
* **Generated Artifacts in** **report/**:
  * `table_describe_raw.csv` &amp; `profile.html` (Data Profiling)
  * `table_cleaning_log.csv` (Audit log of dropped rows)
  * `table_describe_numeric.csv` &amp; `table_describe_categorical.csv`
  * `table_missing.csv` &amp; `fig_eda_missing.png`
  * `table_columns.csv` (Feature inclusion/exclusion justification)
  * `data/processed/asrs.csv` (Cleaned dataset)

## Bước 0: môi trường và cấu trúc thư mục

Chạy một lần trong Terminal của VS Code, không phải trong ô Python. Sau khi cài xong chọn interpreter `.venv` (Ctrl+Shift+P, *Python: Select Interpreter*).

```bash
# Step 1a: environment and project layout (run once)
python -m venv .venv && source .venv/bin/activate      # Windows: .venv\Scripts\activate
pip install pandas numpy duckdb pyarrow scikit-learn lightgbm xgboost catboost \
    statsmodels shap matplotlib seaborn ydata-profiling mapie jupyter
pip freeze > requirements.txt
mkdir -p data/raw data/processed notebooks sql src report
printf "data/raw\ndata/processed\n.venv\n*.parquet\n" > .gitignore
```

In [ ]:
# Environment check and shared imports for the whole notebook
import pandas as pd, numpy as np, duckdb, matplotlib
import matplotlib.pyplot as plt, pathlib, warnings
from pathlib import Path
import shutil
import io
from scipy.stats import kruskal, spearmanr
import seaborn as sns

current = Path.cwd()
project_root = current
while project_root.name and project_root.name != "Group6_ADY201m_FA26_AI2108":
    if (project_root / "ASRS_FULL_DATASET.csv").exists():
        break
    project_root = project_root.parent

processed_dir = project_root / "data" / "processed"
report_dir = project_root / "report"
warnings.filterwarnings('ignore')
for d in ['data/raw', 'data/processed', 'notebooks', 'sql', 'src', 'report']: (project_root / d).mkdir(parents=True, exist_ok=True)
np.random.seed(42); pd.set_option('display.width', 160); pd.set_option('display.max_columns', 40)

## Bước 1: Nạp dữ liệu


In [ ]:
# Locate raw data directory
possible_paths = [
    Path.cwd().parent / "data" / "raw",
    Path.cwd() / "data" / "raw",
    Path.cwd().parent.parent / "data" / "raw"
]

raw_dir = None
for p in possible_paths:
    if p.exists() and (p / "ASRS_FULL_DATASET.csv").exists():
        raw_dir = p
        break

if not raw_dir:
    root = Path.cwd()
    while root.name and root.name != "Group6_ADY201m_FA26_AI2108":
        root = root.parent

    target_dir = root / "data" / "raw"
    target_dir.mkdir(parents=True, exist_ok=True)

    csv_src = root / "ASRS_FULL_DATASET.csv"
    if csv_src.exists():
        shutil.move(str(csv_src), str(target_dir / "ASRS_FULL_DATASET.csv"))
    raw_dir = target_dir

target_file = raw_dir / "ASRS_FULL_DATASET.csv"

# Load directly with Pandas to handle byte decoding safely
df = pd.read_csv(target_file, encoding='cp1252', encoding_errors='replace', low_memory=False, skiprows=[1])

con = duckdb.connect()
con.execute("CREATE OR REPLACE TABLE feat AS SELECT * FROM df")

# is_uas flag
df['is_uas'] = df.get('Aircraft 1.1', '').astype(str).str.contains('UAS|Unmanned', case=False, na=False) # type: ignore

# Update DuckDB table with processed DataFrame
con.execute("CREATE OR REPLACE TABLE feat AS SELECT * FROM df")

print(f"Số dòng: {df.shape[0]}")
print(f"Số cột: {df.shape[1]}")
print(f"Tỷ lệ UAS: {df['is_uas'].mean() * 100:.1f}%")
date_col = 'date' if 'date' in df.columns else 'date_of_event'
if date_col in df.columns:
    print(f"Khoảng thời gian: Từ {df[date_col].min()} đến {df[date_col].max()}")

print(f"Số đơn vị: {df['id'].nunique() if 'id' in df.columns else df.shape[0]}")

Số dòng: 125812
Số cột: 127
Tỷ lệ UAS: 0.5%
Số đơn vị: 125812


### Hồ sơ tự động

`ydata-profiling` trên mẫu 20 nghìn dòng tạo `report/profile.html`; mở bằng trình duyệt để xem nhanh phân phối, thiếu, tương quan

In [ ]:
df.describe(include='all').T.to_csv(str(report_dir / 'table_describe_raw.csv'))

try:
    from ydata_profiling import ProfileReport
    sample = df.sample(min(20000, len(df)), random_state=42)
    ProfileReport(sample, minimal=True).to_file(str(report_dir / 'profile.html'))
except Exception as e:
    profile_info = df.describe(include='all').T.to_html()
    with open(report_dir / 'profile.html', 'w', encoding='utf-8') as f:
        f.write(f"<h1>Data Summary Profile</h1>{profile_info}")

Export report to file: 100%|██████████| 1/1 [00:00<00:00,  9.56it/s]


## Bước 2: Làm sạch, ghép nguồn thứ hai

In [ ]:
# Step 2: clean empty fields, build 6-8 class labels from Anomaly
import re

df.columns = [c.strip().lower().replace(' ', '_').replace('.', '_') for c in df.columns]
# is_uas flag
df['is_uas'] = df.get('aircraft_1_1', '').astype(str).str.contains('UAS|Unmanned', case=False, na=False) # type: ignore

df = df.rename(columns={'acn': 'id', 'report_1_2':'synopsis', 'events':'anomaly', 'aircraft_1_9':'flight_phase', 'place_5':'altitude', 'environment_3':'light','aircraft_1_7':'mission','environment':'weather', 'aircraft_1_2':'aircraft_type'})
report_1 = df['report_1'] if 'report_1' in df.columns else pd.Series('', index=df.index)
report_2 = df['report_2'] if 'report_2' in df.columns else pd.Series('', index=df.index)

# vùng trời của Aircraft 1
df['airspace_ac1'] = df['aircraft_1_11'].astype(str).str.extract(r'(Class\s+[A-G])', flags=re.IGNORECASE)[0].str.upper()
# vùng trời của Aircraft 2 (nếu có)
df['airspace_ac2'] = df['aircraft_2_11'].astype(str).str.extract(r'(Class\s+[A-G])', flags=re.IGNORECASE)[0].str.upper()
# Ưu tiên lấy vùng trời có cấp độ rủi ro cao hơn hoặc kết hợp
df['airspace'] = df['airspace_ac1'].fillna(df['airspace_ac2']).fillna('UNKNOWN')

df['narrative'] = (report_1.fillna('').astype(str) + " " + report_2.fillna('').astype(str)).str.strip()
# Xử lý cột narrative an toàn
if 'narrative' in df.columns:
    df = df.dropna(subset=['narrative'])
    df = df[df['narrative'].astype(str).str.len() > 100]

# 3. Lọc trùng lặp an toàn (kiểm tra cột id/acn)
id_col = 'id' if 'id' in df.columns else ('acn' if 'acn' in df.columns else None)
if id_col:
    df = df.drop_duplicates(subset=[id_col])

#  Gán nhãn label
def label(a):
    a = str(a).lower()
    for k in ['airspace violation', 'conflict', 'deviation', 'aircraft equipment']:
        if k in a: return k
    return 'other'

if 'anomaly' in df.columns:
    df['y'] = df['anomaly'].map(label)

# 5. Xử lý cột year/date an toàn
if 'date' in df.columns:
    df['year'] = pd.to_datetime(df['date'], errors='coerce').dt.year
elif 'date_time' in df.columns:
    df['year'] = pd.to_datetime(df['date_time'], errors='coerce').dt.year

# 6. In kết quả và xuất file Parquet
if 'y' in df.columns:
    print(df['y'].value_counts())

output_path = processed_dir / 'asrs.csv'
df.to_csv(str(output_path), index=False)
print(f"Đã lưu dữ liệu đã làm sạch vào: {output_path}")

y
deviation             54238
aircraft equipment    22164
conflict              18602
other                 12092
airspace violation     3892
Name: count, dtype: int64
Đã lưu dữ liệu đã làm sạch vào: c:\Users\ACER\Desktop\AI_process\Group6_ADY201m_FA26_AI2108\data\processed\asrs.csv


### Audit Log table tracking data cleaning operations (Human Delta)

In [ ]:
# 1. Đảm bảo cột 'year' tồn tại an toàn trước khi chạy clean_log
if 'year' not in df.columns:
    if 'time' in df.columns:
        df['year'] = df['time'].astype(str).str[:4].astype(int)
    else:
        df['year'] = 2025  # Giá trị mặc định nếu không tìm thấy cột ngày

# 2. Đảm bảo cột 'y' tồn tại (nếu chưa có)
if 'y' not in df.columns and 'anomaly' in df.columns:
    df['y'] = df['anomaly']

# 3. Hàm clean_log an toàn
def clean_log(df0, steps):
    rows, d = [], df0.copy()
    for name, fn, why in steps:
        n0 = len(d)
        d = fn(d)
        rows.append([name, n0, len(d), n0 - len(d), why])
    return d, pd.DataFrame(rows, columns=['step', 'rows_before', 'rows_after', 'dropped', 'reason'])

# 4. Thực thi log làm sạch
cols_to_drop = [c for c in ["id"] if c in df.columns]
_, cleaning_log = clean_log(df, [
    ('drop duplicates', lambda x: x.drop_duplicates(subset=cols_to_drop), 'same unit and timestamp'),
    ('drop missing target', lambda x: x.dropna(subset=['y']) if 'y' in x.columns else x, 'cannot be forecast'),
])

print(cleaning_log)

                  step  rows_before  rows_after  dropped                   reason
0      drop duplicates       110988      110988        0  same unit and timestamp
1  drop missing target       110988      110988        0       cannot be forecast


### Numerical and Categorical feature profiling

In [ ]:
eda = df.copy()

# 1. Kiểm tra và tạo cột narrative_len an toàn
narrative_col = None
for col in ['narrative','sypnosis']:
    if col in eda.columns:
        narrative_col = col
        break

if narrative_col:
    eda['narrative_len'] = eda[narrative_col].fillna('').astype(str).str.len()
else:
    eda['narrative_len'] = 0

# 3. Thông tin DataFrame
buf = io.StringIO()
eda.info(buf=buf)
print(buf.getvalue()[:3000])

# 4. Thống kê biến số (Numeric)
numeric_cols = eda.select_dtypes('number').columns
if len(numeric_cols) > 0:
    desc_num = eda[numeric_cols].describe().T
    desc_num['skew'] = eda[numeric_cols].skew()
    desc_num.round(3).to_csv(str(report_dir / 'table_describe_numeric.csv'))
    print("--- Numeric Features ---")
    print(desc_num.round(3).head(15))

# 5. Thống kê biến phân loại (Categorical)
cat_cols = eda.select_dtypes(exclude='number').columns
if len(cat_cols) > 0:
    desc_cat = eda[cat_cols].describe().T
    desc_cat.to_csv(str(report_dir / 'table_describe_categorical.csv'))
    print("\n--- Categorical Features ---")
    print(desc_cat.head(15))

# 6. Kiểm tra các bất thường (Notes)
notes = []
for c in numeric_cols:
    s = eda[c].dropna()
    if len(s) == 0:
        continue
    if s.std() == 0:
        notes.append([c, 'constant column, drop'])
    if s.max() in (999, 9999, 99999, -9999):
        notes.append([c, 'agency missing code, replace with NaN'])
    if abs(s.skew()) > 2:
        notes.append([c, 'strongly skewed, consider log or tree models'])

pd.DataFrame(notes, columns=['column', 'note']).to_csv(str(report_dir / 'table_eda_notes.csv'), index=False)
print("\n--- EDA Notes ---")
print(notes)

<class 'pandas.core.frame.DataFrame'>
Index: 110988 entries, 0 to 124425
Columns: 134 entries, id to narrative_len
dtypes: bool(1), float64(1), int64(2), object(130)
memory usage: 113.6+ MB

--- Numeric Features ---
                  count      mean       std     min     25%     50%     75%      max   skew
unnamed:_125        0.0       NaN       NaN     NaN     NaN     NaN     NaN      NaN    NaN
year           110988.0  2014.980     6.013  2002.0  2010.0  2015.0  2020.0   2025.0  0.024
narrative_len  110988.0  1680.207  1378.461   101.0   768.0  1311.0  2162.0  23782.0  2.509

--- Categorical Features ---
                count  unique            top   freq
id             110988  110988        2217046      1
time           110988     255         200807    714
time_1          98984       5      1201-1800  37361
place          105344    4085    ZZZ.Airport  43633
place_1        105347      72             US  55775
place_2          5929     352            270    390
place_3         17021 

### Calculate missingness rates and generate co-missingness heatmap

In [ ]:
# 2. Đảm bảo biến eda tồn tại từ df
if 'eda' not in locals() and 'df' in locals():
    eda = df.copy()

# 3. Phân tích tỷ lệ missing data
miss = eda.isna().mean().sort_values(ascending=False).rename('missing_rate').to_frame()
miss['decision'] = np.select(
    [miss.missing_rate == 0, miss.missing_rate < 0.05, miss.missing_rate < 0.4],
    ['keep', 'interpolate/impute median', 'drop or keep flag'], 
    default='drop column'
)

# Xuất CSV missing rate
output_csv = report_dir / 'table_missing.csv'
miss.round(4).to_csv(str(output_csv))
print(f"Đã lưu: {output_csv}")
print(miss.head(15))

# 4. Phân tích missingness of target by time & unit (nếu có cột y)
if 'y' in eda.columns:
    tcol = 'year' if 'year' in eda.columns else ('date' if 'date' in eda.columns else None)
    gcol = 'is_uas' if 'is_uas' in eda.columns else ('acn' if 'acn' in eda.columns else eda.columns[0])

    if tcol:
        by_time = eda.groupby(pd.to_datetime(eda[tcol], errors='coerce').dt.to_period('M') if not np.issubdtype(eda[tcol].dtype, np.number) else eda[tcol])['y'].apply(lambda s: s.isna().mean())
        print('target missing by period (top):', by_time.sort_values(ascending=False).head(5).round(3).to_dict())

    if gcol:
        by_unit = eda.groupby(gcol)['y'].apply(lambda s: s.isna().mean()).sort_values(ascending=False)
        print('target missing by unit (top):', by_unit.head(5).round(3).to_dict())

# 5. Co-missingness matrix
co = eda.isna().astype(int)
co = co.loc[:, co.sum() > 0]
if co.shape[1] > 1:
    print("\n--- Co-missingness Matrix ---")
    print(co.corr().round(2))

# 6. Vẽ đồ thị và lưu hình ảnh
fig, ax = plt.subplots(figsize=(7, 3))
miss.missing_rate.head(12).plot.bar(ax=ax, color='#1B6B6D')
ax.set_ylabel('Missing rate')
ax.spines[['top', 'right']].set_visible(False)
fig.tight_layout()

output_fig = report_dir / 'fig_eda_missing.png'
fig.savefig(str(output_fig), dpi=300)
plt.show()

print(f"Đã lưu biểu đồ vào: {output_fig}")

Đã lưu: c:\Users\ACER\Desktop\AI_process\Group6_ADY201m_FA26_AI2108\report\table_missing.csv
               missing_rate     decision
aircraft_2_25      1.000000  drop column
unnamed:_125       1.000000  drop column
aircraft_2_21      1.000000  drop column
aircraft_2_3       1.000000  drop column
aircraft_2_13      1.000000  drop column
place_6            1.000000  drop column
aircraft_2_19      1.000000  drop column
aircraft_2_32      0.999991  drop column
aircraft_2_20      0.999982  drop column
aircraft_2_15      0.999982  drop column
aircraft_2_22      0.999982  drop column
aircraft_2_12      0.999982  drop column
aircraft_2_29      0.999973  drop column
aircraft_2_24      0.999964  drop column
aircraft_2_30      0.999964  drop column
target missing by period (top): {2002: 0.0, 2005: 0.0, 2006: 0.0, 2007: 0.0, 2008: 0.0}
target missing by unit (top): {False: 0.0, True: 0.0}

--- Co-missingness Matrix ---
              time_1  place  place_1  place_2  place_3  place_4  altitude  pla